# 单元二：自动求导机制与计算图 (Autograd & Graphs)

---

## 单元 2.1：动态计算图 —— 梯度的“生命周期”

在“鱼书”笔记第 5 章中，你推导过复杂的矩阵微分。在 PyTorch 中，这一切被抽象为一棵**有向无环图 (DAG)**。

### 1. 核心定义：梯度的追踪器

*   **`requires_grad`**：这是一个布尔标记。如果设为 `True`，PyTorch 会开始追踪该张量上的所有运算，并构建计算图。
*   **`grad_fn`**：这是张量的“出生证明”。它记录了该张量是通过什么运算产生的（如 `AddBackward`, `MulBackward`）。
*   **`backward()`**：触发器。调用它时，PyTorch 会根据**链式法则**，从输出节点反向遍历计算图，计算所有叶子节点的梯度。
*   **`.grad`**：累加器。计算出的偏导数 $\frac{\partial out}{\partial x}$ 会存储在这里。

### 2. 实验：追踪一个复合函数

请在 Jupyter 中运行以下代码，观察偏导数的自动计算过程：

In [1]:
import torch

# 1. 创建输入张量，并开启梯度追踪
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([3.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# 2. 构建计算过程 (计算图在这一步动态生成)
# y = w * x + b
y = w * x + b

# 3. 查看 grad_fn
print(f"y 的产生方式 (grad_fn): {y.grad_fn}") 

# 4. 执行反向传播
y.backward()

# 5. 查看偏导数
# dy/dx = w = 3.0
# dy/dw = x = 2.0
# dy/db = 1.0
print(f"dy/dx: {x.grad}")
print(f"dy/dw: {w.grad}")
print(f"dy/db: {b.grad}")

y 的产生方式 (grad_fn): <AddBackward0 object at 0x70e09c418c10>
dy/dx: tensor([3.])
dy/dw: tensor([2.])
dy/db: tensor([1.])


在模块二中，单元 2.1 是整个深度学习框架的“灵魂”所在。你之前看“鱼书”笔记时，第 5 章手推的反向传播逻辑，在 PyTorch 里全部被集成到了这个 **Autograd（自动求导系统）** 中。

为了让你彻底理解 2.1 的实验内容，我们从**数学本质**、**程序逻辑**和**竞赛应用**三个维度进行深度拆解。

---

#### 1. 深度定义解释：Autograd 的“三驾马车”

在实验代码中，这三个属性共同定义了一个张量在计算图中的角色：

*   **`requires_grad=True`（身份标记）**：
    *   **本质**：告诉 PyTorch 的计算引擎：“这个张量是可变的，请为它开辟一块内存空间来记录它的导数（`.grad`），并追踪所有涉及它的运算。”
    *   **竞赛对标**：在光子 AI 模型中，你的权重矩阵 $W$ 必须开启这个标记，否则模型无法根据误差进行更新（学习）。

*   **`grad_fn`（运算溯源/导数函数）**：
    *   **本质**：它是张量的“出生证明”。当你执行 `y = w * x` 时，产生的 `y` 会自带一个 `MulBackward` 属性。
    *   **逻辑**：它存储了如何计算该操作导数的**数学公式**。在反向传播时，PyTorch 并不需要重新推导公式，它只需要查阅 `grad_fn` 里的“导数说明书”，代入数值即可。

*   **`backward()`（连锁反应开关）**：
    *   **本质**：它是**链式法则（Chain Rule）**的程序化触发器。
    *   **动作**：当你对最终结果（通常是 Loss）调用 `.backward()` 时，它会从最后一片叶子开始，顺着 `grad_fn` 像推倒多米诺骨牌一样，把导数一层层传回到最开始的输入张量。

---

#### 2. 实验步骤的逻辑解剖

我们来看那个公式：$y = w \cdot x + b$

##### **第一阶段：前向传播（Forward Pass）与 动态建图**
当你执行代码 `y = w * x + b` 时，PyTorch 内部发生了两件事：
1.  **数值计算**：算出 $3.0 \times 2.0 + 1.0 = 7.0$。
2.  **动态建图**：在内存中瞬间搭建了一棵树（图）：
    *   底端是叶子节点：`w`($3.0$), `x`($2.0$), `b`($1.0$)。
    *   中间节点：`mul` 运算（产生了中间变量 `temp = w * x`）。
    *   顶端节点：`add` 运算（产生了最终变量 `y`）。
    *   **注意**：这棵图是“动态”的，每次运行 `y = ...` 都会重新构建，这比 TensorFlow 的静态图灵活得多。

##### **第二阶段：反向传播（Backward Pass）**
当你执行 `y.backward()` 时，计算引擎开始工作：
1.  它看 `y` 的 `grad_fn`，发现是 `AddBackward`。
2.  根据加法求导法则：$\frac{\partial y}{\partial (w \cdot x)} = 1$ 和 $\frac{\partial y}{\partial b} = 1$。
3.  接着它看中间变量 `temp = w \cdot x` 的 `grad_fn`，发现是 `MulBackward`。
4.  根据乘法求导法则（链式法则）：
    *   $\frac{\partial y}{\partial w} = \frac{\partial y}{\partial temp} \cdot \frac{\partial temp}{\partial w} = 1 \cdot x = 2.0$
    *   $\frac{\partial y}{\partial x} = \frac{\partial y}{\partial temp} \cdot \frac{\partial temp}{\partial x} = 1 \cdot w = 3.0$
5.  **结果存储**：最终算出的 $2.0, 3.0, 1.0$ 分别被塞进了 `w.grad`, `x.grad`, `b.grad` 中。



---

## 单元 2.2：梯度累加与清除

这是 PyTorch 初学者最容易犯错的地方。

*   **累加特性**：在默认情况下，PyTorch **不会**在每次 `backward()` 后清空 `.grad`。如果你多次运行反向传播，梯度会不断叠加。
*   **原因**：这在处理超大规模 Batch（分批梯度累加）时非常有用，但在常规训练中，我们必须在每次迭代前清空梯度。

#### **实验：观察累加现象**

In [2]:
# 再次运行一次反向传播（假设 y 是重新计算的）
y = w * x + b
y.backward()
print(f"第二次运行后的 dy/dx: {x.grad}") # 你会发现结果变成了 6.0 (3.0 + 3.0)

# 清零操作 (在模块五中，优化器 zero_grad() 会自动做这件事)
x.grad.zero_()
print(f"清零后的 dy/dx: {x.grad}")

第二次运行后的 dy/dx: tensor([6.])
清零后的 dy/dx: tensor([0.])
